# Semiconductor Manufacturing Fault Detection using the SECOM Dataset

This notebook builds a classical machine learning workflow for predicting SECOM pass/fail outcomes from high-dimensional manufacturing process parameters.


![Electronics manufacturing line](../reports/figures/electronics_manufacturing.png)

*Illustrative view of electronic devices being assembled in a clean manufacturing environment.*


## Contents

1. [Introduction](#1-introduction)
2. [Vocabulary](#2-vocabulary)
3. [Problem Formulation](#3-problem-formulation)
4. [Dataset Description](#4-dataset-description)
5. [Exploratory Data Analysis](#5-exploratory-data-analysis)
6. [Missing Values and Data Quality](#6-missing-values-and-data-quality)
7. [Preprocessing Strategy](#7-preprocessing-strategy)
8. [Baseline Models](#8-baseline-models)
9. [Handling Class Imbalance](#9-handling-class-imbalance)
10. [Model Evaluation](#10-model-evaluation)
11. [PCA / Dimensionality Reduction](#11-pca--dimensionality-reduction)
12. [Discussion and Limitations](#12-discussion-and-limitations)
13. [Conclusion](#13-conclusion)


## 1. Introduction

Semiconductor manufacturing produces many numerical measurements during each process run. In this notebook, one **process run** means one row / manufacturing observation in the SECOM dataset.

The project goal is to predict whether a process run passes or fails quality control while handling missing values, noisy process parameters, and strong class imbalance. The workflow uses classical machine learning models and keeps preprocessing leakage-safe.


## 2. Vocabulary

Before working with the dataset, it is useful to clarify the main terms used in this notebook.

- **Process run**: one row in the SECOM dataset. It represents one manufacturing observation collected during production.
- **Process parameter**: one numerical measurement from the manufacturing process. For example, in a factory producing electronic devices, temperature, pressure, alignment, machine speed, or an electrical test reading could all be process parameters.
- **Target label**: the value the model tries to predict. In this project, it indicates whether the process run passed or failed quality control.
- **SECOM label convention**: `1` means fail/faulty and `-1` means pass/normal.
- **Pass sample**: a process run labeled as normal.
- **Fail sample**: a process run labeled as faulty.
- **Class imbalance**: a situation where one class is much more common than the other. In this dataset, pass samples are much more common than fail samples.


## 3. Problem Formulation

This is a binary classification task. For each process run, the model receives a vector of process parameter measurements and predicts whether the run passes or fails quality control.

Mathematically, let $x_i \in \mathbb{R}^p$ be the process parameter vector for process run $i$, where $p$ is the number of measured parameters. Let $y_i \in \{-1, 1\}$ be the quality label, where `1` means Fail / faulty and `-1` means Pass / normal.

The goal is to learn a function $f(x_i)$ that generalizes to unseen process runs, with special attention to detecting the rare Fail class.


## 4. Dataset Description

The SECOM dataset contains anonymized numerical measurements from semiconductor manufacturing. Each row is a process run, each feature column is a process parameter, and the target label indicates the quality outcome.

The original label convention is important: `1` means Fail / faulty and `-1` means Pass / normal. Because the features are anonymized, the analysis focuses on missingness, variance, target imbalance, and predictive performance rather than physical interpretation of individual parameters.


## 5. Exploratory Data Analysis

This section loads the SECOM data, combines the feature matrix with the target labels, and inspects the dataset before modeling. The main EDA goals are to understand class imbalance, missing values, constant parameters, and other data quality issues that affect preprocessing.

The findings from this section guide the preprocessing strategy used later in the modeling pipeline.


In [ ]:
# Imports
from pathlib import Path

# Basic tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn modules
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


In [ ]:
# Define paths to the data files
DATA_DIR = Path("../data/raw")

features_path = DATA_DIR / "secom.data"
labels_path = DATA_DIR / "secom_labels.data"

# Load the data
X_raw = pd.read_csv(features_path, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_path, sep=r"\s+", header=None)

# Check the shapes of the loaded data
X_raw.shape, y_raw.shape

In [ ]:
# Display the first few rows of the features
X_raw.head()

In [ ]:
# Display the first few rows of the labels
y_raw.head()

In [ ]:
# Let's create a clean dataframe by concatenating the features and labels
feature_columns = [f"parameter_{i}" for i in range(X_raw.shape[1])]

X = X_raw.copy()
X.columns = feature_columns

labels = y_raw.copy()
labels.columns = ["target", "timestamp"]

df = X.copy()
df["target"] = labels["target"]
df["timestamp"] = labels["timestamp"]

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Let's check for target class distribution
# In SECOM, 1 = Fail and -1 = Pass.
class_order = [1, -1]
class_labels = ["Fail (1)", "Pass (-1)"]

target_counts = df["target"].value_counts().reindex(class_order, fill_value=0)
target_percentages = df["target"].value_counts(normalize=True).reindex(class_order, fill_value=0) * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

# Create a bar plot for the target distribution
plt.figure(figsize=(6, 4))

bars = plt.bar(
    class_labels,
    target_summary["count"],
    width=0.25,
    color=["darkorange", "steelblue"]
)

# Add value labels on top of the bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    percentage = target_summary.iloc[i]["percentage"]

    # Add the count label
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

    # Add the percentage label inside the bar
    plt.text(
        bar.get_x() + bar.get_width() / 2,   # center of the bar on X
        (height / 2) - 10,                   # center of the bar on Y with a small offset
        f"{percentage:.1f}%",                # only the percentage
        ha="center",
        va="center",
        color="white",
        fontsize=12,
        fontweight="bold"
    )

# Set the title and labels for the plot
plt.title("Target Distribution: Fail vs Pass")
plt.xlabel("Target label")
plt.ylabel("Count")

plt.ylim(0, target_summary["count"].max() * 1.1)
plt.show()


The target distribution shows strong class imbalance: Pass samples (`-1`) represent `93.4%` of the data, while Fail samples (`1`) represent only `6.6%`.

This means accuracy alone can be misleading. A model that mostly predicts Pass may look accurate while still missing many failed process runs, so later evaluation should focus on Fail-class recall, precision, F1-score, confusion matrix, and PR-AUC.


## 6. Missing Values and Data Quality

SECOM contains missing values across many process parameters, so this section checks missingness from several angles. Feature-level missingness asks which parameters have missing values, row-level missingness asks whether some process runs have unusually many missing parameter values, and target-class missingness compares average missingness between Pass and Fail samples.

This section also checks for constant parameters that add no predictive information. Any preprocessing decisions based on missingness or feature quality should later be fit on the training data only, preferably inside a `scikit-learn` Pipeline.


In [ ]:
# Separate process parameter measurements from the target and timestamp columns
feature_cols = [col for col in df.columns if col not in ["target", "timestamp"]]
X_features = df[feature_cols]

print(f"Number of process runs: {X_features.shape[0]}")
print(f"Number of process parameters: {X_features.shape[1]}")


In [ ]:
# Overall missingness across the feature matrix
total_missing = X_features.isna().sum().sum()
total_values = X_features.size
overall_missing_percentage = total_missing / total_values * 100

missing_overview = pd.DataFrame({
    "total_values": [total_values],
    "missing_values": [total_missing],
    "missing_percentage": [overall_missing_percentage]
})

missing_overview


The dataset has missing values, but the global missing percentage only gives a broad overview. A low overall rate can still hide a small group of process parameters with very poor coverage.

The next step is therefore to inspect missingness at the feature level.


In [ ]:
# Missing values by feature
missing_summary = pd.DataFrame({
    "missing_count": X_features.isna().sum(),
    "missing_percentage": X_features.isna().mean() * 100
})

missing_summary = missing_summary.sort_values("missing_percentage", ascending=False)

missing_summary.head(20)


In [ ]:
# Count how many features exceed useful missingness thresholds
missing_thresholds = [0, 5, 10, 20, 40, 60, 80]

missing_threshold_summary = pd.DataFrame({
    "missing_threshold_percentage": missing_thresholds,
    "number_of_features_above_threshold": [
        (missing_summary["missing_percentage"] > threshold).sum()
        for threshold in missing_thresholds
    ]
})

missing_threshold_summary


In [ ]:
# Visualize the features with the highest missing value percentages
top_missing = missing_summary.head(20).sort_values("missing_percentage")

plt.figure(figsize=(8, 6))
bars = plt.barh(
    top_missing.index,
    top_missing["missing_percentage"],
    color="steelblue"
)

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center"
    )

plt.title("Top 20 Features by Missing Value Percentage")
plt.xlabel("Missing values (%)")
plt.ylabel("Feature")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


Feature-level missingness is uneven. Many process parameters have at least one missing value, and several parameters are missing for more than `40%` of process runs. A few are missing for more than `80%`, which makes them weak candidates for direct modeling.

A reasonable preprocessing strategy is to remove parameters with extremely high missingness and impute the remaining missing values. These choices should be learned from the training split only.


In [ ]:
# Missing values by process run / row
row_missing_summary = pd.DataFrame({
    "missing_count": X_features.isna().sum(axis=1),
    "missing_percentage": X_features.isna().mean(axis=1) * 100,
    "target": df["target"]
})

row_missing_summary[["missing_count", "missing_percentage"]].describe()


In [ ]:
# Distribution of missing values per process run
plt.figure(figsize=(7, 4))

plt.hist(
    row_missing_summary["missing_percentage"],
    bins=30,
    color="steelblue",
    edgecolor="black"
)

plt.axvline(
    row_missing_summary["missing_percentage"].mean(),
    color="darkorange",
    linestyle="--",
    label="Mean"
)

plt.axvline(
    row_missing_summary["missing_percentage"].median(),
    color="black",
    linestyle=":",
    label="Median"
)

plt.title("Distribution of Missing Values per Process Run")
plt.xlabel("Missing values per row (%)")
plt.ylabel("Number of process runs")
plt.legend()
plt.tight_layout()
plt.show()


The row-level missingness distribution shows that most process runs have a relatively low missing percentage, roughly around `3-6%`. The mean is slightly higher than the median, which suggests a right-skewed distribution caused by a small number of process runs with unusually high missingness.

A few observations have more than `10%` missing values, with extreme cases above `20%`.


In [ ]:
# Compare the average percentage of missing parameter values by target class
target_order = [1, -1]
target_labels = ["Fail (1)", "Pass (-1)"]

row_missing_by_target = (
    row_missing_summary
    .groupby("target")[["missing_count", "missing_percentage"]]
    .mean()
    .reindex(target_order)
)

row_missing_by_target.index = target_labels
row_missing_by_target


In [ ]:
# Plot average missingness by target class
plt.figure(figsize=(6, 4))

bars = plt.bar(
    row_missing_by_target.index,
    row_missing_by_target["missing_percentage"],
    width=0.35,
    color=["darkorange", "steelblue"]
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}%",
        ha="center",
        va="bottom"
    )

plt.title("Average Missing Values by Target Class")
plt.xlabel("Target class")
plt.ylabel("Average missing values per row (%)")
plt.ylim(0, row_missing_by_target["missing_percentage"].max() * 1.2)
plt.tight_layout()
plt.show()


The row-level view checks whether missing data is concentrated in a small number of process runs. The target-class comparison then checks whether Pass and Fail samples have noticeably different average missingness.

In this dataset, average missingness is similar across target classes, so missingness appears to be a general data quality issue rather than an obvious direct separator between Pass and Fail outcomes.


In [ ]:
# Detect constant features, which do not help classification
unique_counts = X_features.nunique(dropna=True)
constant_features = unique_counts[unique_counts <= 1]

print(f"Number of constant features: {len(constant_features)}")

constant_features.head(20)

In [ ]:
# Compact data quality summary for preprocessing decisions
data_quality_summary = pd.DataFrame({
    "metric": [
        "features_with_any_missing_values",
        "features_with_more_than_40_percent_missing",
        "features_with_more_than_80_percent_missing",
        "constant_features"
    ],
    "value": [
        (missing_summary["missing_percentage"] > 0).sum(),
        (missing_summary["missing_percentage"] > 40).sum(),
        (missing_summary["missing_percentage"] > 80).sum(),
        len(constant_features)
    ]
})

data_quality_summary


This data quality check motivates the preprocessing strategy for the next section. The dataset should not be passed directly into a model without handling missing values and uninformative columns.

A realistic first plan is to remove parameters with very high missingness, remove constant features, impute the remaining missing values, and scale features when required by the model. All learned preprocessing steps should be fit on training data only.


## 7. Preprocessing Strategy

Preprocessing is necessary because SECOM has hundreds of anonymized numerical parameters, missing values, constant or near-constant features, and a strongly imbalanced binary target.

The workflow first defines the feature matrix `X` and target vector `y`, then creates a stratified train/test split. Stratification is important because the Fail class is rare, so both splits should preserve the original class ratio as much as possible.

Later model pipelines will handle imputation, feature filtering, scaling, and modeling. The key leakage rule is that learned preprocessing must be fit only on training data or inside cross-validation folds.


The split below creates the objects used by later modeling code. No imputer, scaler, feature filter, PCA step, or model should be fitted before this split.


In [ ]:
# Define features and target
X = df[feature_cols]
y = df["target"]

# Create a stratified train/test split before fitting any preprocessing step
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape, y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)


## 8. Baseline Models

This section starts with a naive Dummy Classifier and then compares it with the current Logistic Regression baseline. The goal is to separate two questions: what performance can be achieved without learning useful patterns, and how much the first real model improves on that reference point.

Both models are evaluated on the same held-out test set, with `1` treated as the Fail class and `-1` treated as the Pass class.

### 8.1 Dummy Model

The Dummy Classifier is a naive benchmark. With `strategy="prior"`, it learns only the class distribution from the training data and predicts the majority class for every test sample.

This model is not expected to detect failures. Its purpose is to show how misleading accuracy can be when the Pass class dominates the dataset.

In [ ]:
# Dummy model: learns only the class distribution
dummy_model = DummyClassifier(strategy="prior", random_state=42)
dummy_model.fit(X_train, y_train)

dummy_pred = dummy_model.predict(X_test)

# In SECOM, 1 = Fail and -1 = Pass.
dummy_fail_class_index = list(dummy_model.classes_).index(1)
dummy_fail_proba = dummy_model.predict_proba(X_test)[:, dummy_fail_class_index]

print(classification_report(
    y_test,
    dummy_pred,
    labels=[1, -1],
    target_names=["Fail (1)", "Pass (-1)"],
    zero_division=0
))

### 8.2 Logistic Regression Baseline

Logistic Regression is the first real baseline model. It is simple, fast, and a common starting point for binary classification.

The model is wrapped in a `scikit-learn` Pipeline because Logistic Regression cannot handle missing values directly and is sensitive to feature scale. The pipeline applies median imputation, removes constant features, scales the remaining parameters, and fits a class-weighted Logistic Regression model.

In [ ]:
# Logistic Regression baseline pipeline
# scikit-learn adjusts class weights inversely proportional to class frequencies.
log_reg_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("variance_filter", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_pipeline.fit(X_train, y_train)

log_reg_pred = log_reg_pipeline.predict(X_test)

# In SECOM, 1 = Fail and -1 = Pass.
log_reg_fail_class_index = list(log_reg_pipeline.named_steps["classifier"].classes_).index(1)
log_reg_fail_proba = log_reg_pipeline.predict_proba(X_test)[:, log_reg_fail_class_index]

# Keep these aliases for the evaluation cells below.
y_pred = log_reg_pred
y_fail_proba = log_reg_fail_proba

### 8.3 Baseline Evaluation and Comparison

The next cells inspect the Logistic Regression confusion matrix and classification report, then compare the Dummy Classifier and Logistic Regression using the same metrics.

In [ ]:
# Confusion matrix for Logistic Regression baseline predictions
# In SECOM, 1 = Fail and -1 = Pass.
cm = confusion_matrix(y_test, log_reg_pred, labels=[1, -1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fail (1)", "Pass (-1)"]
)

disp.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(
    y_test,
    log_reg_pred,
    labels=[1, -1],
    target_names=["Fail (1)", "Pass (-1)"]
))

In [ ]:
roc_auc = roc_auc_score(y_test == 1, log_reg_fail_proba)
pr_auc = average_precision_score(y_test == 1, log_reg_fail_proba)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")

In [ ]:
def summarize_baseline_model(model_name, y_true, y_pred, y_fail_proba):
    """Return the main metrics for the Fail class and overall model comparison."""
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "fail_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "fail_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "fail_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_true == 1, y_fail_proba),
        "pr_auc": average_precision_score(y_true == 1, y_fail_proba),
    }

baseline_comparison = pd.DataFrame([
    summarize_baseline_model("Dummy Classifier", y_test, dummy_pred, dummy_fail_proba),
    summarize_baseline_model("Logistic Regression", y_test, log_reg_pred, log_reg_fail_proba),
]).set_index("model")

baseline_comparison.round(3)

### 8.4 Baseline Evaluation Conclusion

The Dummy Classifier is useful because it shows what happens when the model mostly follows the majority Pass class. It can achieve high accuracy, but it has `0.00` recall for the Fail class because it does not identify failed process runs.

The Logistic Regression baseline improves on the dummy model by detecting some Fail samples. However, Fail-class performance is still weak: precision is `0.11`, recall is `0.19`, F1-score is `0.14`, ROC-AUC is `0.644`, and PR-AUC is `0.124`.

The main lesson is that accuracy is not enough for this dataset. The next section should focus on imbalance-aware strategies that improve Fail-class detection while keeping false alarms manageable.

## 9. Handling Class Imbalance

Fail samples are rare, so accuracy can hide poor failure detection. This section will compare imbalance-aware strategies such as class weights, threshold adjustment, and metrics focused on the minority class.

The main goal is to improve Fail-class recall while keeping false alarms at a reasonable level.


## 10. Model Evaluation

Model evaluation will use cross-validation on the training set and a held-out test set for final evaluation. The main metrics will include confusion matrix, precision, recall, F1-score, ROC-AUC, and PR-AUC.

For this fault detection task, Fail-class recall is especially important, but it must be balanced against false alarms.


## 11. PCA / Dimensionality Reduction

Because SECOM has hundreds of process parameters, PCA may help reduce noise and create lower-dimensional representations for some models.

If PCA is used, it must be placed inside a Pipeline after imputation and scaling, so the transformation is learned only from training data or cross-validation folds.


## 12. Discussion and Limitations

This section will interpret model results and compare the trade-off between detecting failures and avoiding false alarms. It should also explain which metrics are most meaningful for this imbalanced fault detection problem.

Important limitations include anonymized process parameters, a small number of Fail samples, missing values, and limited ability to infer causal process explanations.


## 13. Conclusion

The final conclusion will summarize the best-performing model, the most useful evaluation metrics, the main lessons from the analysis, and recommendations for future work.

The final project should demonstrate a complete, readable, leakage-safe classical ML workflow suitable for course review.
